<a href="https://colab.research.google.com/github/adeeljames/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/adeeljames/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


Section 1: Two signal checks
Signal A — Staleness (linked to FlyRank's refresh flags)

In [2]:
# Bucket by staleness, show decline rate per bucket
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 10000],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)

staleness_check = df.groupby("staleness_bucket").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
)
print(staleness_check)

                      n  decline_rate
staleness_bucket                     
<90d              20655      0.512031
90-180d            9171      0.611057
180-365d            169      0.467456
365d+                 5      0.600000


/tmp/ipykernel_1516/2021749571.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_check = df.groupby("staleness_bucket").agg(


Signal A: Staleness (days_since_last_update) — linked to FlyRank's refresh
flags (stale_visible_page).

Bucket results:
  <90d:      n=20,655  decline_rate=0.512
  90-180d:   n=9,171   decline_rate=0.611
  180-365d:  n=169     decline_rate=0.467
  365d+:     n=5       decline_rate=0.600

Verdict: MIXED

Reasoning: decline rate does rise from <90d (0.512) to 90-180d (0.611), which
is the direction the "stale = declining" belief predicts. But it does NOT
continue rising for 180-365d (drops to 0.467) or clearly hold for 365d+ (only
n=5, too small to trust). The relationship is not a clean staircase, and the
two oldest buckets have very little data (169 and 5 rows) compared to the
first two buckets (20,655 and 9,171) — so I can't confidently say staleness
alone predicts decline beyond the 90-180 day range. This is a clearly-
explained negative/mixed result, and it means my baseline rule's staleness
threshold (>=180 days) should be treated as a starting guess, not a proven
cutoff — worth revisiting once I have more data in the 180+ day range.

Signal B — CTR vs position tier (linked to CTR-fix logic)

In [3]:
visible = df[df["impressions_90d"] >= 100]
ctr_check = visible.groupby("position_tier").agg(
    n=("ctr", "size"),
    avg_ctr=("ctr", "mean")
).sort_values("avg_ctr", ascending=False)
print(ctr_check)

                  n   avg_ctr
position_tier                
page_1         8633  0.354760
top_3           533  0.334128
striking       5903  0.255782
page_3_5       6058  0.142359
deep            879  0.055415


Signal B: CTR vs position tier — linked to FlyRank's CTR-fix logic
(low_ctr_visible_page).

Verdict: Signal B: CTR vs position tier — linked to FlyRank's CTR-fix logic
(low_ctr_visible_page).

Bucket results:
  page_1:    n=8,633  avg_ctr=0.355
  top_3:     n=533    avg_ctr=0.334
  striking:  n=5,903  avg_ctr=0.256
  page_3_5:  n=6,058  avg_ctr=0.142
  deep:      n=879    avg_ctr=0.055

Verdict: CONFIRMED

Reasoning: CTR drops smoothly and consistently as position tier worsens —
from 0.355 at page_1 down to 0.055 at deep, with a clear, monotonic decline
across every tier and reasonably large sample sizes in each bucket (except
top_3, which is smaller at n=533). This strongly confirms the "CTR collapses
as position worsens" belief, which is exactly the logic behind FlyRank's
CTR-fix flag — any CTR comparison in my baseline or later model MUST be done
within the same position tier, never across tiers, or it will be comparing
apples to oranges.

Section 2: Encode ONE rule (score + reason code + action)

In [4]:
# ONE simple rule: stale + visible pages get flagged
df["baseline_score"] = (
    (df["days_since_last_update"] >= 180).astype(int) *
    (df["impressions_90d"] >= 500).astype(int) *
    df["impressions_90d"]
)

df["reason_code"] = "stale_visible_page"
df["action"] = "refresh"

queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "client_id", "baseline_score", "reason_code", "action",
     "impressions_90d", "days_since_last_update", "trend_direction"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved", len(queue), "rows to work/outputs/baseline_action_score.csv")
queue.head(10)

Saved 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,refresh,61678,194,down
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,refresh,59472,194,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,refresh,25715,194,down
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,refresh,13299,193,down
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,refresh,7812,194,down
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,refresh,7558,193,down
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,refresh,4590,194,down
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,refresh,4556,194,down
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,refresh,4429,194,down
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,refresh,1697,193,down


Section 3: Top-10 review

1. content_cf56e2e2e282 — action: refresh. Why: stale (194 days since update) AND
   highly visible (61,678 impressions/90d), trend already declining. Would be
   wrong if: this traffic drop is due to a sibling page on the same site
   absorbing the demand (consolidation) rather than genuine content decay.

2. content_7368877ea310 — action: refresh. Why: same client, same staleness
   window (194 days), very high visibility (59,472 impressions). Would be
   wrong if: this is a seasonal dip that will recover on its own without
   intervention.

3. content_1bfaa38ff26c — action: refresh. Why: stale (194 days), strong
   visibility (25,715 impressions), declining trend. Would be wrong if: the
   decline is driven by a SERP layout change (e.g. a featured snippet stealing
   clicks) rather than the content itself being outdated.

4. content_0a91db491d14 — action: refresh. Why: stale (193 days), solid
   visibility (13,299 impressions), declining. Would be wrong if: impressions
   are inflated by a broad/irrelevant keyword match that doesn't reflect real
   search intent.

5. content_5feee3994adb — action: refresh. Why: stale (194 days), moderate-high
   visibility (7,812 impressions), declining. Would be wrong if: the page was
   already scheduled for a redesign/migration, making a content refresh
   redundant work.

6. content_c2d929d83eaa — action: refresh. Why: stale (193 days), moderate
   visibility (7,558 impressions), declining. Would be wrong if: this page is
   near end-of-life/low business priority despite the traffic numbers.

7. content_b16bd7307b39 — action: refresh. Why: stale (194 days), moderate
   visibility (4,590 impressions), declining. Would be wrong if: the decline
   is small/noisy and within normal week-to-week variance rather than a real
   trend.

8. content_fe16a55cd13d — action: refresh. Why: stale (194 days), moderate
   visibility (4,556 impressions), declining. Would be wrong if: engagement
   metrics (not shown in this rule) are actually healthy, meaning the page
   doesn't need a content rewrite.

9. content_ecb6215e79fd — action: refresh. Why: stale (194 days), moderate
   visibility (4,429 impressions), declining. Would be wrong if: this is a
   duplicate/near-duplicate of another higher-priority page already in the
   queue.

10. content_928af3e22c80 — action: refresh. Why: stale (193 days), lower but
    still meaningful visibility (1,697 impressions), declining. Would be wrong
    if: 1,697 impressions is too low relative to this client's typical page
    performance to justify reviewer time over higher-volume candidates.

Section 4 — Weak picks

The clearest weakness in this top-10 is CLIENT CONCENTRATION: all ten rows
belong to the same single client (client_7f2253d7e2), and all share nearly
identical staleness (193-194 days since last update). This strongly suggests
this client had a single bulk content-update event around the same date,
rather than ten independently-declining pages. As a rule, this means the
baseline score -- driven mostly by raw impression volume -- is letting one
large, high-traffic client dominate the entire top of the queue, starving
other clients' legitimately declining pages of review time.

Row 10 (content_928af3e22c80) is the weakest individual pick: its impressions
(1,697) are an order of magnitude lower than rows 1-3, meaning it's only in
the top 10 because of this client's dominance, not because it's genuinely a
stronger candidate than declining pages from other clients.

Fix for later weeks: either normalize the score within each client, or cap
how many pages per client can appear in the same top-K review batch.

Section 5 — Self-check

- [x] Two signal checks done, bucket tables with n shown, verdicts given
- [x] One signal linked to a real FlyRank flag (staleness -> refresh flags)
- [x] One rule encoded: score + ONE reason code + action label
- [x] Ranked queue written to work/outputs/baseline_action_score.csv (30,000 rows)
- [x] Top 10 reviewed, one line each: action, why, what would make it wrong
- [x] Named a weak pattern (single-client concentration) — no future-window or
      label-derived inputs used